In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Treceri de Transpiler alimentate de AI

Trecerile de Transpiler alimentate de AI sunt treceri care funcționează ca înlocuitor direct al trecerilor Qiskit „tradiționale" pentru unele sarcini de transpilare. Acestea produc adesea rezultate mai bune decât algoritmii heuristici existenți (cum ar fi adâncime mai mică și număr mai mic de CNOT), dar sunt și mult mai rapide decât algoritmii de optimizare precum rezolvatorii de satisfiabilitate booleană. Trecerile de Transpiler AI rulează în mediul tău local.

> **Note:** Trecerile de Transpiler alimentate de AI se află în stadiu de lansare beta și pot fi modificate.
>     Dacă ai feedback sau dorești să contactezi echipa de dezvoltare, te rog să folosești acest [canal Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Următoarele treceri sunt disponibile în prezent:

**Treceri de rutare**
 - `AIRouting`: Selectarea layout-ului și rutarea Circuit-ului

**Treceri de sinteză a Circuit-ului**
 - `AICliffordSynthesis`: Sinteza Circuit-ului Clifford
 - `AILinearFunctionSynthesis`: Sinteza Circuit-ului de funcție liniară
 - `AIPermutationSynthesis`: Sinteza Circuit-ului de permutare

Pentru a utiliza trecerile de Transpiler AI, mai întâi instalează pachetul `qiskit-ibm-transpiler`. Vizitează [documentația API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) pentru mai multe informații despre diferitele opțiuni disponibile.

## Rulează trecerile de Transpiler AI local sau în cloud
Mai întâi instalează `qiskit-ibm-transpiler` cu câteva dependențe suplimentare, astfel:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService
import logging

backend = QiskitRuntimeService().backend("ibm_fez")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)
transpiled_circuit = ai_passmanager.run(circuit)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

După instalarea dependențelor suplimentare, modul implicit de rulare a trecerilor de Transpiler alimentate de AI este să folosești mașina ta locală.

## Trecerea de rutare AI
Trecerea `AIRouting` acționează atât ca etapă de layout, cât și ca etapă de rutare. Poate fi folosită într-un `PassManager` astfel:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit.circuit.library import efficient_su2

ibm_kingston = QiskitRuntimeService().backend("ibm_kingston")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_kingston,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_kingston, local_mode=True
        ),  # Re-synthesize Linear Function blocks
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fetching 127 files:   0%|          | 0/127 [00:00<?, ?it/s]

The synthesis respects the coupling map of the device: it can be run safely after other routing passes without disturbing the circuit, so the overall circuit will still follow the device restrictions. By default, the synthesis will replace the original sub-circuit only if the synthesized sub-circuit improves the original (currently only checking CNOT count), but this can be forced to always replace the circuit by setting `replace_only_if_better=False`.

The following synthesis passes are available from `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis for [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) circuits (blocks of `H`, `S`, and `CX` gates). Currently up to nine qubit blocks.
- *AILinearFunctionSynthesis*: Synthesis for [Linear Function](/docs/api/qiskit/qiskit.circuit.library.LinearFunction) circuits (blocks of `CX` and `SWAP` gates). Currently up to nine qubit blocks.
- *AIPermutationSynthesis*: Synthesis for [Permutation](/docs/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuits (blocks of `SWAP` gates). Currently available for 65, 33, and 27 qubit blocks.
- *AIPauliNetworkSynthesis*: Synthesis for Pauli Network circuits (blocks of `H`, `S`, `SX`, `CX`, `RX`, `RY` and `RZ` gates). Currently up to six qubit blocks.

We expect to gradually increase the size of the supported blocks.

All passes use a thread pool to send several requests in parallel. By default, the number for max threads is the number of cores plus four (default values for the `ThreadPoolExecutor` Python object). However, you can set your own value with the `max_threads` argument at pass instantiation. For example, the following line instantiates the `AILinearFunctionSynthesis` pass, which allows it to use a maximum of 20 threads.

```python
AILinearFunctionSynthesis(backend=ibm_torino, max_threads=20)  # Re-synthesize Linear Function blocks using 20 threads max
```

You can also set the environment variable `AI_TRANSPILER_MAX_THREADS` to the desired number of maximum threads, and all synthesis passes instantiated after that will use that value.

For the AI synthesis passes to synthesize a sub-circuit, it must lay on a connected subgraph of the coupling map (one way to do this is with a routing pass before collecting the blocks, but this is not the only way to do it). The synthesis passes will automatically check that the specific subgraph is supported, and if not, it will raise a warning and leave the original sub-circuit unchanged.

The following custom collection passes for Cliffords, Linear Functions and Permutations that can be imported from `qiskit_ibm_transpiler.ai.collection` also complement the synthesis passes:

- *CollectCliffords*: Collects Clifford blocks as `Instruction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectLinearFunctions*: Collects blocks of `SWAP` and `CX` as `LinearFunction` objects and stores the original sub-circuit to compare against it after synthesis.
- *CollectPermutations*: Collects blocks of `SWAP` circuits as `Permutations`.
- *CollectPauliNetworks*: Collects Pauli Network blocks and stores the original sub-circuit to compare against it after synthesis.

These custom collection passes limit the sizes of the collected sub-circuits so they are supported by the AI-powered synthesis passes. Therefore, it is recommended to use them after the routing passes and before the synthesis passes for a better overall optimization.

## Hybrid heuristic-AI circuit transpilation

The `qiskit-ibm-transpiler` allows you to configure a hybrid pass manager that combines the best of Qiskit's heuristic and the AI-powered transpiler passes. This feature behaves similarly to the Qiskit `generate_pass_manager` method. A typical way to use `generate_ai_pass_manager` is as follows:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_kingston")
kingston_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=kingston_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Aici, `backend` determină harta de cuplare pentru care se face rutarea, `optimization_level` (1, 2 sau 3) determină efortul de calcul depus în proces (o valoare mai mare produce de obicei rezultate mai bune, dar durează mai mult), iar `layout_mode` specifică modul de gestionare a selecției layout-ului.
`layout_mode` include următoarele opțiuni:

- `keep`: Aceasta respectă layout-ul setat de trecerile anterioare ale Transpiler-ului (sau folosește layout-ul trivial dacă nu este setat). Este utilizat de obicei doar atunci când Circuit-ul trebuie rulat pe Qubiți specifici ai dispozitivului. Produce adesea rezultate mai slabe deoarece are mai puțin spațiu pentru optimizare.
- `improve`: Aceasta folosește layout-ul setat de trecerile anterioare ale Transpiler-ului ca punct de plecare. Este utilă atunci când ai o estimare inițială bună pentru layout; de exemplu, pentru Circuit-uri construite într-un mod care urmează aproximativ harta de cuplare a dispozitivului. Este de asemenea utilă dacă vrei să încerci alte treceri de layout specifice combinate cu trecerea `AIRouting`.
- `optimize`: Acesta este modul implicit. Funcționează cel mai bine pentru Circuit-uri generale unde s-ar putea să nu ai estimări bune de layout. Acest mod ignoră selecțiile anterioare de layout.
- `local_mode`: Acest indicator determină unde rulează trecerea `AIRouting`. Dacă este `False`, `AIRouting` rulează de la distanță prin Qiskit Transpiler Service. Dacă este `True`, pachetul încearcă să ruleze trecerea în mediul tău local, cu o rezervă la modul cloud dacă dependențele necesare nu sunt găsite.

## Treceri de sinteză a Circuit-ului AI
Trecerile de sinteză a Circuit-ului AI îți permit să optimizezi bucăți din diferite tipuri de circuit-uri ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Funcție Liniară](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutare](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Rețea Pauli) prin re-sintetizarea lor. Un mod tipic de a folosi trecerea de sinteză este următorul: